# 06 Build ML Forecast Panel

This notebook assembles the final store-category-day forecasting panel for the machine-learning notebooks. It combines the realised-weather master sales panel with the operational weather-feature view from notebook 05 and writes both the ML panel and the feature registry. The notebook performs data assembly only and does not train models.


## Pipeline position

This notebook follows `notebooks/05_weather_ensemble_features.ipynb` and precedes `notebooks/07_ml_benchmark_models.ipynb`. Its main weather input is `data/processed/weather_forecast_operational_ml_features.parquet`. Deprecated scaffold, sigma, and simulated-baseline forecast files are not part of the active path.


## Inputs, outputs, and model contract

Inputs are the realised-weather master sales panel, the notebook 05 operational weather view, and the notebook 05 weather feature metadata. Outputs are `data/processed/ml_forecast_panel_full.parquet`, `outputs/ml_panel/ml_panel_feature_registry.csv`, and compact notebook 06 audit CSVs.

M1 contains no weather. M2 adds operational point forecast weather only. M3 is the realised-weather perfect-information benchmark. M4 adds calibrated uncertainty, probability, and interval weather features. `apparent_temp_fcst_mean` is excluded from all active M1-M4 feature lists.


## Setup

The setup defines project-root discovery, input/output paths, shared helpers, and package imports for panel construction.


In [ ]:
import gc
from pathlib import Path

import numpy as np
import pandas as pd

MARKER_FILES = ("README_FOR_CODEX.md", "AGENTS.md", "pyproject.toml")

def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if any((candidate / marker).exists() for marker in MARKER_FILES):
            return candidate
    raise FileNotFoundError("Could not find the project root from the current working directory.")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_ML_PANEL_DIR = OUTPUT_DIR / "ml_panel"
OUTPUT_WEATHER_FEATURE_DIR = OUTPUT_DIR / "weather_ensemble_features"

MASTER_PANEL_PATH = PROCESSED_DIR / "master_panel_realized_weather.parquet"
WEATHER_FEATURES_PATH = PROCESSED_DIR / "weather_forecast_operational_ml_features.parquet"
WEATHER_FEATURE_GROUPS_PATH = OUTPUT_WEATHER_FEATURE_DIR / "05_ml_weather_feature_groups.csv"

ML_PANEL_PATH = PROCESSED_DIR / "ml_forecast_panel_full.parquet"
FEATURE_REGISTRY_PATH = OUTPUT_ML_PANEL_DIR / "ml_panel_feature_registry.csv"
FEATURE_REGISTRY_AUDIT_PATH = OUTPUT_ML_PANEL_DIR / "06_model_feature_registry.csv"
INPUT_SCHEMA_AUDIT_PATH = OUTPUT_ML_PANEL_DIR / "06_input_schema_audit.csv"
SHAPE_SUMMARY_PATH = OUTPUT_ML_PANEL_DIR / "06_ml_panel_shape_summary.csv"
HORIZON_SUMMARY_PATH = OUTPUT_ML_PANEL_DIR / "06_horizon_summary.csv"
MISSINGNESS_BY_GROUP_PATH = OUTPUT_ML_PANEL_DIR / "06_missingness_by_model_feature_group.csv"
LEAKAGE_COLUMN_CHECK_PATH = OUTPUT_ML_PANEL_DIR / "06_leakage_column_check.csv"
FORECAST_SOURCE_SUMMARY_PATH = OUTPUT_ML_PANEL_DIR / "06_forecast_source_summary.csv"
PANEL_CHECKS_PATH = OUTPUT_ML_PANEL_DIR / "ml_panel_checks.csv"
PANEL_MISSINGNESS_PATH = OUTPUT_ML_PANEL_DIR / "ml_panel_missingness.csv"
PANEL_TARGET_SUMMARY_PATH = OUTPUT_ML_PANEL_DIR / "ml_panel_target_summary.csv"
PANEL_COLUMN_SUMMARY_PATH = OUTPUT_ML_PANEL_DIR / "ml_panel_column_summary.csv"
PANEL_LEAKAGE_CHECKS_PATH = OUTPUT_ML_PANEL_DIR / "ml_panel_sample_leakage_checks.csv"

for folder in [PROCESSED_DIR, OUTPUT_ML_PANEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

def project_relative(path):
    path = Path(path)
    try:
        return path.relative_to(PROJECT_ROOT).as_posix()
    except ValueError:
        return str(path)

def require_file(path, description):
    if not Path(path).exists():
        raise FileNotFoundError(f"Missing {description}: {project_relative(path)}")

def require_columns(frame, required_columns, frame_name):
    missing = [column for column in required_columns if column not in frame.columns]
    if missing:
        raise KeyError(f"{frame_name} is missing required columns: {missing}")

def report_memory(frame, name):
    memory_gb = frame.memory_usage(deep=True).sum() / 1_000_000_000
    print(f"{name}: shape={frame.shape}, approx_memory={memory_gb:.3f} GB")
    return memory_gb

def parquet_metadata(path):
    path = Path(path)
    if not path.exists():
        return {
            "exists": False,
            "rows": pd.NA,
            "columns": pd.NA,
            "size_mb": pd.NA,
            "column_names": [],
            "schema_error": "",
        }
    try:
        import pyarrow.parquet as pq
        parquet_file = pq.ParquetFile(path)
        return {
            "exists": True,
            "rows": parquet_file.metadata.num_rows,
            "columns": len(parquet_file.schema_arrow.names),
            "column_names": parquet_file.schema_arrow.names,
            "size_mb": round(path.stat().st_size / 1_000_000, 2),
            "schema_error": "",
        }
    except Exception as exc:
        return {
            "exists": True,
            "rows": pd.NA,
            "columns": pd.NA,
            "column_names": [],
            "size_mb": round(path.stat().st_size / 1_000_000, 2),
            "schema_error": f"{type(exc).__name__}: {exc}",
        }

def to_bool(series):
    if str(series.dtype) == "bool":
        return series
    return series.astype(str).str.strip().str.lower().isin(["true", "1", "yes"])

print(f"Project root: {PROJECT_ROOT}")
print(f"Final ML panel output: {project_relative(ML_PANEL_PATH)}")

## Configuration

The configuration defines forecast horizons, target dates, key columns, weather-feature groups, historical feature windows, and feature-registry roles used downstream.


In [ ]:
HORIZONS = list(range(0, 11))
MAIN_OPERATIONAL_WEATHER_WINDOW = "trade_08_19"
DROP_CATEGORY = "Ekskluderes"
SALES_LAG_DAYS = [1, 7, 14, 28]
SALES_ROLL_WINDOWS = [7, 14, 28]
SAME_WEEKDAY_LAGS = [1, 2, 3, 4]
CAMPAIGN_LAG_DAYS = [1, 7, 14]
CAMPAIGN_ROLL_WINDOWS = [7, 28]
EXPECTED_M2_WEATHER_FEATURES = ["temp_fcst_mean", "precip_fcst_sum", "wind_fcst_mean", "humid_fcst_mean", "cloud_fcst_mean"]
EXPECTED_M4_EXTRA_FEATURES = [
    "temp_fcst_std", "temp_fcst_p10", "temp_fcst_p90", "temp_fcst_interval_width",
    "wind_fcst_std", "wind_fcst_p10", "wind_fcst_p90", "wind_fcst_interval_width",
    "humid_fcst_std", "humid_fcst_p10", "humid_fcst_p90", "humid_fcst_interval_width",
    "cloud_fcst_std", "cloud_fcst_p10", "cloud_fcst_p90", "cloud_fcst_interval_width",
    "precip_wet_prob", "precip_amount_uncertainty", "precip_fcst_p10", "precip_fcst_p90",
    "precip_fcst_interval_width", "precip_wet_amount_p90", "cloud_open_prob", "cloud_partly_prob", "cloud_overcast_prob",
]
MASTER_PANEL_COLUMNS = [
    "DatoSolgt", "AvdelingID", "AvdelingTekst", "Region", "Analyse_Kategori", "Ukedag", "Årstid",
    "Helligdager", "HelligdagNavn", "Antall_total", "log_Salg", "Avdeling_total", "Closed",
    "CampaignDay", "CampaignShare", "Antall_campaign", "Antall_ordinary",
    "precip_val", "temp_mean", "humidity_mean", "wind_mean", "cloud_mean",
]
TARGET_COLUMNS = ["Antall_total", "log_Salg"]
REALISED_TARGET_DAY_WEATHER_COLUMNS = ["precip_val", "temp_mean", "humidity_mean", "wind_mean", "cloud_mean"]
TARGET_DATE_CAMPAIGN_COLUMNS = ["CampaignDay", "CampaignShare", "Antall_campaign", "Antall_ordinary"]
FORECAST_AUDIT_METADATA_COLUMNS = [
    "forecast_source", "forecast_support_level", "calibration_source", "uncertainty_source",
    "calibration_fallback_level", "uncertainty_fallback_level", "climatology_fallback_level",
    "h2_anchor_weight", "climatology_weight", "uncertainty_scale",
]
PANEL_KEY_COLUMNS = [
    "AvdelingID", "Analyse_Kategori", "target_date", "origin_date", "origin_hour",
    "origin_datetime_utc", "horizon_days", "horizon", "aggregation_window",
]
FINAL_DUPLICATE_KEY_COLUMNS = [
    "AvdelingID", "Analyse_Kategori", "target_date", "origin_date", "origin_hour", "origin_datetime_utc", "horizon_days",
]
print(f"Horizons retained: {HORIZONS}")
print(f"Main operational weather window: {MAIN_OPERATIONAL_WEATHER_WINDOW}")

## Input schema checks

The next cell verifies required input files and schemas before loading the modelling panel components.


In [ ]:
for path, description in [
    (MASTER_PANEL_PATH, "master sales/calendar/realised-weather panel from notebook 01"),
    (WEATHER_FEATURES_PATH, "operational weather forecast ML feature view from notebook 05"),
    (WEATHER_FEATURE_GROUPS_PATH, "weather feature-group metadata from notebook 05"),
]:
    require_file(path, description)

input_status_rows = []
for path, description, required in [
    (MASTER_PANEL_PATH, "master sales/calendar/realised-weather panel from notebook 01", MASTER_PANEL_COLUMNS),
    (WEATHER_FEATURES_PATH, "operational weather forecast ML feature view from notebook 05", []),
]:
    meta = parquet_metadata(path)
    available = set(meta.get("column_names", []))
    missing = [column for column in required if column not in available]
    input_status_rows.append(
        {
            "file": project_relative(path),
            "description": description,
            "exists": meta.get("exists", False),
            "size_mb": meta.get("size_mb", pd.NA),
            "rows": meta.get("rows", pd.NA),
            "columns": meta.get("columns", pd.NA),
            "missing_required_columns": ", ".join(missing),
            "schema_error": meta.get("schema_error", ""),
        }
    )

feature_groups = pd.read_csv(WEATHER_FEATURE_GROUPS_PATH)
for flag in [
    "allowed_m2_operational_point_weather",
    "allowed_m4_operational_uncertainty_weather",
    "known_at_origin",
    "leakage_risk",
    "retained_in_ml_weather_view",
]:
    feature_groups[flag] = to_bool(feature_groups[flag])
require_columns(
    feature_groups,
    [
        "column",
        "role",
        "feature_group",
        "allowed_m2_operational_point_weather",
        "allowed_m4_operational_uncertainty_weather",
        "known_at_origin",
        "leakage_risk",
        "retained_in_ml_weather_view",
    ],
    "feature_groups",
)

M2_WEATHER_FEATURES = feature_groups.loc[
    feature_groups["role"].eq("feature") & feature_groups["allowed_m2_operational_point_weather"],
    "column",
].tolist()
M4_WEATHER_FEATURES = feature_groups.loc[
    feature_groups["role"].eq("feature") & feature_groups["allowed_m4_operational_uncertainty_weather"],
    "column",
].tolist()
M4_EXTRA_WEATHER_FEATURES = [column for column in M4_WEATHER_FEATURES if column not in M2_WEATHER_FEATURES]
if M2_WEATHER_FEATURES != EXPECTED_M2_WEATHER_FEATURES:
    raise AssertionError(f"M2 weather features from notebook 05 do not match the contract: {M2_WEATHER_FEATURES}")
missing_m4 = [column for column in EXPECTED_M4_EXTRA_FEATURES if column not in M4_EXTRA_WEATHER_FEATURES]
if missing_m4:
    raise AssertionError(f"M4 extra weather features missing from notebook 05 metadata: {missing_m4}")
if "apparent_temp_fcst_mean" in set(feature_groups["column"]):
    raise AssertionError("apparent_temp_fcst_mean must not be present in notebook 05 ML feature metadata.")

weather_meta = parquet_metadata(WEATHER_FEATURES_PATH)
WEATHER_FEATURE_COLUMNS = feature_groups.loc[feature_groups["retained_in_ml_weather_view"], "column"].tolist()
missing_weather_columns = [
    column for column in WEATHER_FEATURE_COLUMNS if column not in set(weather_meta.get("column_names", []))
]
if missing_weather_columns:
    raise AssertionError(f"Notebook 05 metadata references missing weather columns: {missing_weather_columns}")
input_status_rows.append(
    {
        "file": project_relative(WEATHER_FEATURE_GROUPS_PATH),
        "description": "weather feature-group metadata from notebook 05",
        "exists": True,
        "size_mb": round(WEATHER_FEATURE_GROUPS_PATH.stat().st_size / 1_000_000, 4),
        "rows": len(feature_groups),
        "columns": len(feature_groups.columns),
        "missing_required_columns": "",
        "schema_error": "",
    }
)
input_status = pd.DataFrame(input_status_rows)
if input_status["missing_required_columns"].astype(str).ne("").any():
    raise KeyError("Required input columns are missing. See 06_input_schema_audit.csv.")
print(input_status.to_string(index=False))
print(f"M2 weather features: {M2_WEATHER_FEATURES}")
print(f"M4 weather features: {len(M4_WEATHER_FEATURES)} total")

## Sales calendar panel

This step loads the realised-weather master sales panel columns needed for target construction, calendar controls, opening status, category identifiers, and M3 realised-weather benchmark features.


In [ ]:
master_panel = pd.read_parquet(MASTER_PANEL_PATH, columns=MASTER_PANEL_COLUMNS)
require_columns(master_panel, MASTER_PANEL_COLUMNS, "master_panel")
master_rows_before_filter = len(master_panel)
master_panel = master_panel.loc[master_panel["Analyse_Kategori"].astype(str) != DROP_CATEGORY].copy()
master_rows_after_filter = len(master_panel)
master_panel = master_panel.rename(columns={"DatoSolgt": "target_date"})
master_panel["target_date"] = pd.to_datetime(master_panel["target_date"]).dt.normalize()
master_panel["AvdelingID"] = pd.to_numeric(master_panel["AvdelingID"], errors="raise").astype("int64")
for column in ["Analyse_Kategori", "AvdelingTekst", "Region", "Ukedag", "Årstid"]:
    master_panel[column] = master_panel[column].astype("string")
master_panel["HelligdagNavn"] = master_panel["HelligdagNavn"].fillna("No holiday").astype("string")
master_panel["Closed"] = master_panel["Closed"].astype("int8")
master_panel["is_open"] = (1 - master_panel["Closed"].astype("int16")).astype("int8")
master_panel["target_year"] = master_panel["target_date"].dt.year.astype("int16")
master_panel["target_month"] = master_panel["target_date"].dt.month.astype("int8")
master_panel["target_iso_week"] = master_panel["target_date"].dt.isocalendar().week.astype("int8")
season_map = {
    12: "winter",
    1: "winter",
    2: "winter",
    3: "spring",
    4: "spring",
    5: "spring",
    6: "summer",
    7: "summer",
    8: "summer",
    9: "autumn",
    10: "autumn",
    11: "autumn",
}
master_panel["season"] = master_panel["target_month"].map(season_map).astype("string")
duplicate_master = int(master_panel.duplicated(["AvdelingID", "Analyse_Kategori", "target_date"]).sum())
if duplicate_master:
    raise AssertionError(f"Master panel has {duplicate_master} duplicate keys")
print(f"Master rows before dropping {DROP_CATEGORY!r}: {master_rows_before_filter:,}")
print(f"Master rows after dropping {DROP_CATEGORY!r}: {master_rows_after_filter:,}")
print(f"target_date range: {master_panel['target_date'].min().date()} -> {master_panel['target_date'].max().date()}")
report_memory(master_panel, "master_panel")

## Operational forecast-weather features

This step loads the notebook 05 operational weather feature view and its metadata, preserving the separation between M2 point-weather features, M4 uncertainty features, and audit-only columns.


In [ ]:
weather_features = pd.read_parquet(WEATHER_FEATURES_PATH, columns=WEATHER_FEATURE_COLUMNS)
require_columns(weather_features, WEATHER_FEATURE_COLUMNS, "weather_features")
weather_features["target_date"] = pd.to_datetime(weather_features["target_date"]).dt.normalize()
weather_features["origin_date"] = pd.to_datetime(weather_features["origin_date"]).dt.normalize()
weather_features["origin_datetime_utc"] = pd.to_datetime(weather_features["origin_datetime_utc"], utc=True)
weather_features["AvdelingID"] = pd.to_numeric(weather_features["AvdelingID"], errors="raise").astype("int64")
weather_features["origin_hour"] = pd.to_numeric(weather_features["origin_hour"], errors="raise").astype("int8")
weather_features["horizon_days"] = pd.to_numeric(weather_features["horizon_days"], errors="raise").astype("int8")
weather_features["horizon"] = weather_features["horizon_days"].astype("int8")
for column in [
    "aggregation_window",
    "forecast_source",
    "forecast_support_level",
    "calibration_source",
    "uncertainty_source",
    "calibration_fallback_level",
    "uncertainty_fallback_level",
    "climatology_fallback_level",
]:
    weather_features[column] = weather_features[column].astype("string")
if sorted(weather_features["aggregation_window"].astype(str).unique().tolist()) != [MAIN_OPERATIONAL_WEATHER_WINDOW]:
    raise AssertionError("Weather feature view is not filtered to the main operational window")
weather_horizons = sorted(int(x) for x in weather_features["horizon_days"].unique())
if weather_horizons != HORIZONS:
    raise AssertionError(f"Expected horizons {HORIZONS}, found {weather_horizons}")
weather_key_columns = [
    "origin_date",
    "origin_hour",
    "origin_datetime_utc",
    "target_date",
    "horizon_days",
    "AvdelingID",
    "aggregation_window",
]
duplicate_weather_keys = int(weather_features.duplicated(weather_key_columns).sum())
if duplicate_weather_keys:
    raise AssertionError(f"Weather feature view has {duplicate_weather_keys} duplicate keys")
origin_horizon_diff = (weather_features["target_date"] - weather_features["origin_date"]).dt.days.astype("int16")
if int((origin_horizon_diff != weather_features["horizon_days"].astype("int16")).sum()):
    raise AssertionError("target_date - origin_date differs from horizon_days in weather feature view")
if int(weather_features[M2_WEATHER_FEATURES].isna().sum().sum()) or int(
    weather_features[M4_WEATHER_FEATURES].isna().sum().sum()
):
    raise AssertionError("Weather feature view contains missing M2/M4 values")
source_by_horizon = (
    weather_features.groupby("horizon_days", observed=True)["forecast_source"]
    .agg(lambda s: ", ".join(sorted(set(map(str, s)))))
    .to_dict()
)
expected_source_by_horizon = {
    h: ("deterministic_meps" if h <= 2 else "synthetic_realised_anchor_error_calibrated")
    for h in HORIZONS
}
if source_by_horizon != expected_source_by_horizon:
    raise AssertionError(f"Unexpected forecast_source by horizon: {source_by_horizon}")
print(f"Weather feature rows: {len(weather_features):,}")
print(
    f"Weather target_date range: {weather_features['target_date'].min().date()} "
    f"-> {weather_features['target_date'].max().date()}"
)
report_memory(weather_features, "weather_features")

## Origin-safe historical features

Historical sales and campaign features are keyed by `origin_date` and use only information strictly before the forecast origin. This keeps h=0 rows usable without copying target-day sales into the feature set.


In [ ]:
history_daily = master_panel[[
    "AvdelingID",
    "Analyse_Kategori",
    "target_date",
    "Antall_total",
    "CampaignShare",
    "CampaignDay",
]].copy()
history_daily = history_daily.sort_values(["AvdelingID", "Analyse_Kategori", "target_date"]).reset_index(drop=True)
group_cols = ["AvdelingID", "Analyse_Kategori"]
group_obj = history_daily.groupby(group_cols, sort=False, observed=True)
for offset in SALES_LAG_DAYS:
    history_daily[f"sales_lag_{offset}_origin"] = group_obj["Antall_total"].shift(offset).astype("float32")
for window in SALES_ROLL_WINDOWS:
    history_daily[f"sales_roll_mean_{window}_origin"] = (
        group_obj["Antall_total"]
        .transform(lambda s, w=window: s.shift(1).rolling(w, min_periods=w).mean())
        .astype("float32")
    )
    history_daily[f"sales_roll_std_{window}_origin"] = (
        group_obj["Antall_total"]
        .transform(lambda s, w=window: s.shift(1).rolling(w, min_periods=w).std())
        .astype("float32")
    )
for offset in CAMPAIGN_LAG_DAYS:
    history_daily[f"campaign_share_lag_{offset}_origin"] = group_obj["CampaignShare"].shift(offset).astype("float32")
    history_daily[f"campaign_day_lag_{offset}_origin"] = group_obj["CampaignDay"].shift(offset).astype("float32")
for window in CAMPAIGN_ROLL_WINDOWS:
    history_daily[f"campaign_share_roll_mean_{window}_origin"] = (
        group_obj["CampaignShare"]
        .transform(lambda s, w=window: s.shift(1).rolling(w, min_periods=w).mean())
        .astype("float32")
    )
    history_daily[f"campaign_day_roll_mean_{window}_origin"] = (
        group_obj["CampaignDay"]
        .transform(lambda s, w=window: s.shift(1).rolling(w, min_periods=w).mean())
        .astype("float32")
    )
history_feature_columns = [c for c in history_daily.columns if c.endswith("_origin")]
history_features = history_daily[
    ["AvdelingID", "Analyse_Kategori", "target_date", *history_feature_columns]
].rename(columns={"target_date": "origin_date"})
REQUIRED_HISTORY_FEATURES = [
    "sales_lag_1_origin", "sales_lag_7_origin", "sales_lag_14_origin", "sales_lag_28_origin",
    "sales_roll_mean_7_origin", "sales_roll_mean_14_origin", "sales_roll_mean_28_origin",
    "sales_roll_std_7_origin", "sales_roll_std_14_origin", "sales_roll_std_28_origin",
    "campaign_share_lag_1_origin", "campaign_share_lag_7_origin", "campaign_share_lag_14_origin",
    "campaign_day_lag_1_origin", "campaign_day_lag_7_origin", "campaign_day_lag_14_origin",
    "campaign_share_roll_mean_7_origin", "campaign_share_roll_mean_28_origin",
    "campaign_day_roll_mean_7_origin", "campaign_day_roll_mean_28_origin",
]
print(f"history_features rows: {len(history_features):,}")
print(f"history feature columns: {len(history_feature_columns)}")
report_memory(history_features, "history_features")

## ML panel merge

This step merges sales targets, forecast-weather features, and origin-safe historical features into the final forecast panel keyed by store, category, origin date, target date, and horizon.


In [ ]:
ml_panel = master_panel.merge(
    weather_features,
    on=["AvdelingID", "target_date"],
    how="inner",
    validate="many_to_many",
)
rows_after_sales_weather_merge = len(ml_panel)
ml_panel = ml_panel.merge(
    history_features,
    on=["AvdelingID", "Analyse_Kategori", "origin_date"],
    how="left",
    validate="many_to_one",
)

sales_lookup = master_panel[["AvdelingID", "Analyse_Kategori", "target_date", "Antall_total"]].rename(
    columns={"target_date": "lookup_date", "Antall_total": "lookup_sales"}
)
sales_lookup_indexed = sales_lookup.set_index(["AvdelingID", "Analyse_Kategori", "lookup_date"])["lookup_sales"].sort_index()
base_offsets = (((ml_panel["horizon_days"].astype("int16") - 1).clip(lower=0) // 7) + 1) * 7
same_weekday_safety = []
for lag_number in SAME_WEEKDAY_LAGS:
    offset_days = base_offsets + (lag_number - 1) * 7
    lookup_dates = ml_panel["target_date"] - pd.to_timedelta(offset_days, unit="D")
    same_weekday_safety.append(bool((lookup_dates <= ml_panel["origin_date"]).all()))
    lookup_frame = pd.DataFrame(
        {
            "AvdelingID": ml_panel["AvdelingID"].to_numpy(),
            "Analyse_Kategori": ml_panel["Analyse_Kategori"].astype("string").to_numpy(),
            "lookup_date": lookup_dates.to_numpy(),
        }
    )
    lookup_index = pd.MultiIndex.from_frame(lookup_frame)
    ml_panel[f"sales_same_weekday_lag_{lag_number}_origin"] = sales_lookup_indexed.reindex(
        lookup_index
    ).to_numpy(dtype="float32")
    del lookup_frame, lookup_index, lookup_dates
same_weekday_columns = [f"sales_same_weekday_lag_{lag}_origin" for lag in SAME_WEEKDAY_LAGS]
ml_panel["sales_same_weekday_mean_4_origin"] = ml_panel[same_weekday_columns].mean(axis=1).astype("float32")
REQUIRED_HISTORY_FEATURES = REQUIRED_HISTORY_FEATURES + same_weekday_columns + ["sales_same_weekday_mean_4_origin"]
if not all(same_weekday_safety):
    raise AssertionError("At least one same-target-weekday lookup date is later than origin_date")
rows_before_history_filter = len(ml_panel)
missing_required = ml_panel[REQUIRED_HISTORY_FEATURES].isna().any(axis=1)
rows_dropped_by_history_filter = int(missing_required.sum())
ml_panel = ml_panel.loc[~missing_required].reset_index(drop=True)
rows_after_history_filter = len(ml_panel)
print(f"Rows after sales x operational-weather merge: {rows_after_sales_weather_merge:,}")
print(f"Rows before history filter: {rows_before_history_filter:,}")
print(f"Rows dropped by origin-history burn-in: {rows_dropped_by_history_filter:,}")
print(f"Rows after history filter: {rows_after_history_filter:,}")
report_memory(ml_panel, "ml_panel")
del history_daily, history_features, sales_lookup, sales_lookup_indexed, weather_features
gc.collect()

## Dtypes and column order

The panel is cast to stable dtypes and ordered consistently before output, reducing storage size and keeping downstream schemas reproducible.


In [ ]:
float32_columns = [
    *M2_WEATHER_FEATURES,
    *M4_EXTRA_WEATHER_FEATURES,
    *REALISED_TARGET_DAY_WEATHER_COLUMNS,
    *[c for c in ml_panel.columns if c.endswith("_origin")],
    "CampaignShare",
    "Antall_campaign",
    "Antall_ordinary",
    "Avdeling_total",
    "Antall_total",
]
for column in float32_columns:
    if column in ml_panel.columns:
        ml_panel[column] = pd.to_numeric(ml_panel[column], errors="coerce").astype("float32")
for column in [
    "Closed",
    "is_open",
    "Helligdager",
    "CampaignDay",
    "origin_hour",
    "horizon",
    "horizon_days",
    "target_month",
    "target_iso_week",
]:
    if column in ml_panel.columns:
        ml_panel[column] = pd.to_numeric(ml_panel[column], errors="coerce").astype("int8")
if "target_year" in ml_panel.columns:
    ml_panel["target_year"] = pd.to_numeric(ml_panel["target_year"], errors="coerce").astype("int16")
for column in [
    "AvdelingTekst",
    "Region",
    "Analyse_Kategori",
    "Ukedag",
    "Årstid",
    "HelligdagNavn",
    "season",
    "aggregation_window",
    "forecast_source",
    "forecast_support_level",
    "calibration_source",
    "uncertainty_source",
    "calibration_fallback_level",
    "uncertainty_fallback_level",
    "climatology_fallback_level",
]:
    if column in ml_panel.columns:
        ml_panel[column] = ml_panel[column].astype("string")
front_columns = [
    "target_date",
    "origin_date",
    "origin_datetime_utc",
    "origin_hour",
    "horizon_days",
    "horizon",
    "AvdelingID",
    "Analyse_Kategori",
    "aggregation_window",
    "forecast_source",
    "forecast_support_level",
    "AvdelingTekst", "Region", "Ukedag", "season", "Årstid", "Helligdager", "HelligdagNavn",
    "target_year", "target_month", "target_iso_week", "Antall_total", "log_Salg", "Avdeling_total", "Closed", "is_open",
    *TARGET_DATE_CAMPAIGN_COLUMNS,
    *REALISED_TARGET_DAY_WEATHER_COLUMNS,
    *M2_WEATHER_FEATURES,
    *M4_EXTRA_WEATHER_FEATURES,
    *FORECAST_AUDIT_METADATA_COLUMNS,
]
history_columns_ordered = sorted([c for c in ml_panel.columns if c.endswith("_origin")])
ordered_columns = []
for column in [*front_columns, *history_columns_ordered]:
    if column in ml_panel.columns and column not in ordered_columns:
        ordered_columns.append(column)
ml_panel = ml_panel[ordered_columns + [column for column in ml_panel.columns if column not in ordered_columns]]
print(f"Final column count before registry: {ml_panel.shape[1]}")
report_memory(ml_panel, "ml_panel optimized")

## Feature registry

The feature registry assigns each column to its modelling role and records whether it is allowed in M1, M2, M3, or M4. It also marks known-at-origin status and leakage-risk fields for downstream model construction.


In [ ]:
def registry_row(
    column,
    role,
    feature_group,
    known_at_origin,
    m1=False,
    m2=False,
    m3=False,
    m4=False,
    leakage_risk=False,
    notes="",
):
    return {
        "column": column,
        "role": role,
        "feature_group": feature_group,
        "known_at_origin": bool(known_at_origin),
        "allowed_m1_baseline": bool(m1),
        "allowed_m2_synthetic_point_weather": bool(m2),
        "allowed_m3_perfect_information": bool(m3),
        "allowed_m4_synthetic_engineered_weather": bool(m4),
        "leakage_risk": bool(leakage_risk),
        "notes": notes,
    }

registry_rows = []
for column in PANEL_KEY_COLUMNS:
    registry_rows.append(
        registry_row(
            column,
            "key",
            "key",
            True,
            notes=(
                "Panel key or forecast-origin identifier; retained for joins, audit, "
                "stratification, and horizon evaluation."
            ),
        )
    )
registry_rows.append(
    registry_row(
        "Antall_total",
        "target",
        "target",
        False,
        leakage_risk=True,
        notes="Main ML target on the original quantity scale.",
    )
)
registry_rows.append(
    registry_row(
        "log_Salg",
        "robustness_target",
        "target",
        False,
        leakage_risk=True,
        notes="Log(1 + Antall_total) target retained for robustness only.",
    )
)
for column, note in [
    ("Ukedag", "Target weekday."),
    ("Helligdager", "Holiday indicator."),
    ("HelligdagNavn", "Holiday name."),
    ("season", "English target-date season."),
    ("target_year", "Target-date year."),
    ("target_month", "Target-date month."),
    ("target_iso_week", "Target-date ISO week."),
]:
    registry_rows.append(
        registry_row(
            column,
            "feature",
            "calendar",
            True,
            m1=True,
            m2=True,
            m3=True,
            m4=True,
            notes=note + " Calendar known in advance.",
        )
    )
registry_rows.append(
    registry_row(
        "Årstid",
        "diagnostic",
        "diagnostic_only",
        True,
        notes="Norwegian season label retained for traceability; canonical ML season feature is season.",
    )
)
for column, note in [("AvdelingTekst", "Store name."), ("Region", "Store region.")]:
    registry_rows.append(
        registry_row(
            column,
            "feature",
            "store_category",
            True,
            m1=True,
            m2=True,
            m3=True,
            m4=True,
            notes=note,
        )
    )
for column, note in [("Closed", "Operational opening-status proxy."), ("is_open", "1 - Closed.")]:
    registry_rows.append(
        registry_row(
            column,
            "feature",
            "opening_status",
            True,
            m1=True,
            m2=True,
            m3=True,
            m4=True,
            notes=note,
        )
    )
registry_rows.append(
    registry_row(
        "Avdeling_total",
        "diagnostic",
        "diagnostic_only",
        False,
        leakage_risk=True,
        notes="Realised target-date store total. Not safe as an ML feature.",
    )
)
for column in TARGET_DATE_CAMPAIGN_COLUMNS:
    registry_rows.append(
        registry_row(
            column,
            "diagnostic",
            "raw_campaign_diagnostic",
            False,
            leakage_risk=True,
            notes=(
                "Target-date campaign variable retained for diagnostics only. "
                "Origin-safe historical campaign features are provided separately."
            ),
        )
    )
for column in sorted([c for c in ml_panel.columns if c.startswith("sales_") and c.endswith("_origin")]):
    registry_rows.append(
        registry_row(
            column,
            "feature",
            "historical_sales",
            True,
            m1=True,
            m2=True,
            m3=True,
            m4=True,
            notes="Origin-safe sales-history feature computed strictly before origin_date.",
        )
    )
for column in sorted([c for c in ml_panel.columns if c.startswith("campaign_") and c.endswith("_origin")]):
    registry_rows.append(
        registry_row(
            column,
            "feature",
            "historical_campaign",
            True,
            m1=True,
            m2=True,
            m3=True,
            m4=True,
            notes="Origin-safe historical campaign feature computed strictly before origin_date.",
        )
    )
for column in M2_WEATHER_FEATURES:
    registry_rows.append(
        registry_row(
            column,
            "feature",
            "operational_weather_point",
            True,
            m1=False,
            m2=True,
            m3=False,
            m4=True,
            notes="Operational point forecast weather feature from notebook 05.",
        )
    )
for column in M4_EXTRA_WEATHER_FEATURES:
    registry_rows.append(
        registry_row(
            column,
            "feature",
            "operational_weather_uncertainty_probability_interval",
            True,
            m1=False,
            m2=False,
            m3=False,
            m4=True,
            notes="Operational uncertainty/probability/interval weather feature from notebook 05.",
        )
    )
for column in FORECAST_AUDIT_METADATA_COLUMNS:
    registry_rows.append(
        registry_row(
            column,
            "metadata",
            "forecast_audit_metadata",
            True,
            notes=(
                "Forecast construction/calibration metadata retained for audit and "
                "stratification, not as a default ML feature."
            ),
        )
    )
for column in REALISED_TARGET_DAY_WEATHER_COLUMNS:
    registry_rows.append(
        registry_row(
            column,
            "feature",
            "realised_weather_perfect_information",
            False,
            m1=False,
            m2=False,
            m3=True,
            m4=False,
            leakage_risk=True,
            notes="Realised target-day weather. Not known at origin; allowed only in M3 perfect-information benchmark.",
        )
    )
feature_registry = pd.DataFrame(registry_rows).sort_values(
    ["feature_group", "column"]
).reset_index(drop=True)
extra_in_registry = set(feature_registry["column"]) - set(ml_panel.columns)
missing_in_registry = set(ml_panel.columns) - set(feature_registry["column"])
if extra_in_registry:
    raise AssertionError(f"Registry references columns not present in the panel: {sorted(extra_in_registry)}")
if missing_in_registry:
    raise AssertionError(f"Panel has columns not covered by the registry: {sorted(missing_in_registry)}")
print(f"Registry rows: {len(feature_registry)}")
print(
    feature_registry["feature_group"]
    .value_counts()
    .rename_axis("feature_group")
    .reset_index(name="columns")
    .to_string(index=False)
)

## Diagnostics and leakage checks

These checks summarize panel completeness, duplicate keys, feature availability, target distributions, and leakage guards before the panel is saved.


In [ ]:
duplicate_final_keys = int(ml_panel.duplicated(FINAL_DUPLICATE_KEY_COLUMNS).sum())
if duplicate_final_keys:
    raise AssertionError(f"Final ML panel has {duplicate_final_keys} duplicate keys")
origin_horizon_diff = (ml_panel["target_date"] - ml_panel["origin_date"]).dt.days.astype("int16")
mismatch_final = int((origin_horizon_diff != ml_panel["horizon_days"].astype("int16")).sum())
if mismatch_final:
    raise AssertionError(f"Final panel has {mismatch_final} rows where target_date - origin_date != horizon_days")
if int((ml_panel["horizon"].astype("int16") != ml_panel["horizon_days"].astype("int16")).sum()):
    raise AssertionError("horizon compatibility alias differs from horizon_days")
panel_horizons = sorted(int(x) for x in ml_panel["horizon_days"].unique())
if panel_horizons != HORIZONS:
    raise AssertionError(f"Final panel does not retain h=0..h=10: {panel_horizons}")
m2_missing_final = int(ml_panel[M2_WEATHER_FEATURES].isna().sum().sum())
m4_missing_final = int(ml_panel[M4_WEATHER_FEATURES].isna().sum().sum())
if m2_missing_final or m4_missing_final:
    raise AssertionError(f"Final panel has missing M2/M4 weather values: M2={m2_missing_final}, M4={m4_missing_final}")
if "apparent_temp_fcst_mean" in ml_panel.columns or feature_registry["column"].astype(str).str.contains(
    "apparent", case=False, na=False
).any():
    raise AssertionError("Apparent temperature must be absent from active ML panel feature lists.")
operational_model_flags = [
    "allowed_m1_baseline",
    "allowed_m2_synthetic_point_weather",
    "allowed_m4_synthetic_engineered_weather",
]
operational_features = feature_registry.loc[
    feature_registry[operational_model_flags].any(axis=1) & feature_registry["role"].eq("feature")
]
forbidden_pattern = operational_features["column"].astype(str).str.contains(
    "obs|realised|realized|error|apparent|pressure|mslp",
    case=False,
    regex=True,
    na=False,
)
forbidden_groups = operational_features["feature_group"].isin([
    "realised_weather_perfect_information",
    "raw_campaign_diagnostic",
    "diagnostic_only",
])
if bool((forbidden_pattern | forbidden_groups).any()):
    bad = operational_features.loc[forbidden_pattern | forbidden_groups, ["column", "feature_group"]]
    raise AssertionError(f"Forbidden columns/groups are enabled in an operational model: {bad.to_dict(orient='records')}")
pressure_feature_violations = feature_registry.loc[
    feature_registry["column"].astype(str).str.contains("pressure|mslp", case=False, regex=True, na=False)
    & feature_registry[operational_model_flags].any(axis=1),
    ["column", "feature_group"],
]
if len(pressure_feature_violations):
    raise AssertionError(
        "Pressure/MSLP columns are enabled as operational ML features: "
        f"{pressure_feature_violations.to_dict(orient='records')}"
    )
m2_feature_columns = feature_registry.loc[
    feature_registry["allowed_m2_synthetic_point_weather"] & feature_registry["role"].eq("feature"),
    "column",
].tolist()
if set(c for c in m2_feature_columns if c in M4_WEATHER_FEATURES) != set(M2_WEATHER_FEATURES):
    raise AssertionError("M2 weather feature list is not point-only")
m4_feature_columns = feature_registry.loc[
    feature_registry["allowed_m4_synthetic_engineered_weather"] & feature_registry["role"].eq("feature"),
    "column",
].tolist()
if set(c for c in m4_feature_columns if c in M4_WEATHER_FEATURES) != set(M4_WEATHER_FEATURES):
    raise AssertionError("M4 weather feature list does not match notebook 05 contract")
m3_weather_enabled = feature_registry.loc[
    feature_registry["allowed_m3_perfect_information"]
    & feature_registry["feature_group"].eq("realised_weather_perfect_information"),
    "column",
].tolist()
if set(m3_weather_enabled) != set(REALISED_TARGET_DAY_WEATHER_COLUMNS):
    raise AssertionError(f"M3 realised-weather benchmark features mismatch: {m3_weather_enabled}")
forecast_metadata_allowed = feature_registry.loc[
    feature_registry["column"].isin([
        "forecast_source",
        "forecast_support_level",
        "calibration_fallback_level",
        "uncertainty_fallback_level",
        "climatology_fallback_level",
    ]),
    operational_model_flags,
].any().any()
if bool(forecast_metadata_allowed):
    raise AssertionError("Forecast metadata is enabled as a default ML feature")
source_by_horizon_final = (
    ml_panel.groupby("horizon_days", observed=True)["forecast_source"]
    .agg(lambda s: ", ".join(sorted(set(map(str, s)))))
    .to_dict()
)
expected_source_by_horizon = {
    h: ("deterministic_meps" if h <= 2 else "synthetic_realised_anchor_error_calibrated")
    for h in HORIZONS
}
if source_by_horizon_final != expected_source_by_horizon:
    raise AssertionError(f"Unexpected final forecast_source by horizon: {source_by_horizon_final}")
checks_table = pd.DataFrame([
    {"check": "stores", "value": int(ml_panel["AvdelingID"].nunique())},
    {"check": "active_categories", "value": int(ml_panel["Analyse_Kategori"].nunique())},
    {"check": "horizons", "value": ", ".join(str(int(x)) for x in panel_horizons)},
    {"check": "target_date_min", "value": str(ml_panel["target_date"].min().date())},
    {"check": "target_date_max", "value": str(ml_panel["target_date"].max().date())},
    {"check": "origin_date_min", "value": str(ml_panel["origin_date"].min().date())},
    {"check": "origin_date_max", "value": str(ml_panel["origin_date"].max().date())},
    {"check": "rows_master_before_drop", "value": master_rows_before_filter},
    {"check": "rows_master_after_drop_ekskluderes", "value": master_rows_after_filter},
    {"check": "rows_after_sales_weather_merge", "value": rows_after_sales_weather_merge},
    {"check": "rows_before_history_filter", "value": rows_before_history_filter},
    {"check": "rows_dropped_by_history_filter", "value": rows_dropped_by_history_filter},
    {"check": "rows_after_history_filter", "value": rows_after_history_filter},
    {"check": "duplicate_final_keys", "value": duplicate_final_keys},
    {"check": "origin_horizon_mismatches", "value": mismatch_final},
    {"check": "m2_missing_values", "value": m2_missing_final},
    {"check": "m4_missing_values", "value": m4_missing_final},
    {"check": "registry_columns", "value": int(len(feature_registry))},
    {"check": "saved_panel_columns", "value": int(ml_panel.shape[1])},
])
horizon_summary = (
    ml_panel.groupby(["horizon_days", "forecast_source", "forecast_support_level"], observed=True)
    .size()
    .reset_index(name="rows")
    .sort_values(["horizon_days", "forecast_source"])
)
forecast_source_summary = (
    ml_panel.groupby(["forecast_source", "forecast_support_level"], observed=True)
    .agg(
        rows=("Antall_total", "size"),
        horizons=("horizon_days", lambda s: ", ".join(str(int(x)) for x in sorted(s.unique()))),
        target_date_min=("target_date", "min"),
        target_date_max=("target_date", "max"),
    )
    .reset_index()
)
forecast_source_summary["target_date_min"] = forecast_source_summary["target_date_min"].dt.date.astype(str)
forecast_source_summary["target_date_max"] = forecast_source_summary["target_date_max"].dt.date.astype(str)
missingness = pd.DataFrame(
    {
        "column": ml_panel.columns,
        "non_null_count": [int(ml_panel[c].notna().sum()) for c in ml_panel.columns],
        "missing_count": [int(ml_panel[c].isna().sum()) for c in ml_panel.columns],
        "missing_rate": [float(ml_panel[c].isna().mean()) for c in ml_panel.columns],
    }
)
missingness = missingness.merge(feature_registry[["column", "feature_group", "role"]], on="column", how="left")
missingness_by_group = (
    missingness.groupby(["feature_group", "role"], dropna=False, observed=True)
    .agg(
        columns=("column", "count"),
        total_missing=("missing_count", "sum"),
        max_missing_rate=("missing_rate", "max"),
    )
    .reset_index()
)
leakage_column_check = feature_registry.assign(
    enabled_in_operational_model=feature_registry[operational_model_flags].any(axis=1),
    forbidden_name_pattern=feature_registry["column"].astype(str).str.contains(
        "obs|realised|realized|error|apparent|pressure|mslp",
        case=False,
        regex=True,
        na=False,
    ),
)
column_summary = pd.DataFrame(
    {
        "column": ml_panel.columns,
        "dtype": [str(ml_panel[column].dtype) for column in ml_panel.columns],
        "non_null_count": [int(ml_panel[column].notna().sum()) for column in ml_panel.columns],
    }
).merge(
    feature_registry[["column", "role", "feature_group", "known_at_origin", "leakage_risk"]],
    on="column",
    how="left",
)
target_summary = (
    ml_panel.groupby(["horizon_days", "Analyse_Kategori"], observed=True)
    .agg(
        rows=("Antall_total", "size"),
        zero_share=("Antall_total", lambda s: float((s == 0).mean())),
        closed_share=("Closed", "mean"),
        target_mean=("Antall_total", "mean"),
        target_p50=("Antall_total", lambda s: float(s.median())),
        target_p90=("Antall_total", lambda s: float(s.quantile(0.9))),
    )
    .reset_index()
)
print("Diagnostics passed.")
print(checks_table.to_string(index=False))

## Origin-safety spot check and output write

The final spot check verifies that historical features are aligned to the forecast origin. The notebook then writes the ML panel, feature registry, and audit outputs.


In [ ]:
sample_index = np.linspace(0, len(ml_panel) - 1, num=min(12, len(ml_panel)), dtype=int)
sample_rows = ml_panel.iloc[sample_index][[
    "AvdelingID",
    "Analyse_Kategori",
    "target_date",
    "origin_date",
    "horizon_days",
    "sales_lag_1_origin",
    "sales_lag_7_origin",
    "sales_same_weekday_lag_1_origin",
]].reset_index(drop=True)
master_lookup = master_panel[["AvdelingID", "Analyse_Kategori", "target_date", "Antall_total"]].rename(
    columns={"target_date": "calendar_date"}
)
leakage_records = []
for row in sample_rows.itertuples(index=False):
    origin = row.origin_date
    target = row.target_date
    horizon = int(row.horizon_days)
    avd = int(row.AvdelingID)
    cat = str(row.Analyse_Kategori)
    base_offset = (((max(horizon - 1, 0)) // 7) + 1) * 7
    same_weekday_date = target - pd.Timedelta(days=base_offset)
    mask_base = (master_lookup["AvdelingID"] == avd) & (master_lookup["Analyse_Kategori"].astype(str) == cat)
    expected_lag_1 = master_lookup.loc[
        mask_base & (master_lookup["calendar_date"] == origin - pd.Timedelta(days=1)),
        "Antall_total",
    ]
    expected_lag_7 = master_lookup.loc[
        mask_base & (master_lookup["calendar_date"] == origin - pd.Timedelta(days=7)),
        "Antall_total",
    ]
    expected_sw = master_lookup.loc[mask_base & (master_lookup["calendar_date"] == same_weekday_date), "Antall_total"]
    def scalar(series):
        return float(series.iloc[0]) if len(series) else float("nan")
    leakage_records.append({
        "AvdelingID": avd,
        "Analyse_Kategori": cat,
        "target_date": target.date().isoformat(),
        "origin_date": origin.date().isoformat(),
        "horizon_days": horizon,
        "sales_lag_1_source_date": (origin - pd.Timedelta(days=1)).date().isoformat(),
        "sales_lag_7_source_date": (origin - pd.Timedelta(days=7)).date().isoformat(),
        "same_weekday_source_date": same_weekday_date.date().isoformat(),
        "lag_1_origin_safe": bool(origin - pd.Timedelta(days=1) < origin),
        "lag_7_origin_safe": bool(origin - pd.Timedelta(days=7) < origin),
        "same_weekday_origin_safe": bool(same_weekday_date <= origin),
        "lag_1_match": bool(np.isclose(row.sales_lag_1_origin, scalar(expected_lag_1), equal_nan=True)),
        "lag_7_match": bool(np.isclose(row.sales_lag_7_origin, scalar(expected_lag_7), equal_nan=True)),
        "same_weekday_lag_1_match": bool(np.isclose(row.sales_same_weekday_lag_1_origin, scalar(expected_sw), equal_nan=True)),
    })
leakage_check_df = pd.DataFrame(leakage_records)
for column in [
    "lag_1_origin_safe",
    "lag_7_origin_safe",
    "same_weekday_origin_safe",
    "lag_1_match",
    "lag_7_match",
    "same_weekday_lag_1_match",
]:
    if not bool(leakage_check_df[column].all()):
        failed = leakage_check_df.loc[~leakage_check_df[column]]
        raise AssertionError(f"Origin-safety spot-check failed for {column}: {failed.to_dict(orient='records')}")

ml_panel.to_parquet(ML_PANEL_PATH, index=False)
feature_registry.to_csv(FEATURE_REGISTRY_PATH, index=False)
feature_registry.to_csv(FEATURE_REGISTRY_AUDIT_PATH, index=False)
input_status.to_csv(INPUT_SCHEMA_AUDIT_PATH, index=False)
checks_table.to_csv(PANEL_CHECKS_PATH, index=False)
checks_table.to_csv(SHAPE_SUMMARY_PATH, index=False)
horizon_summary.to_csv(HORIZON_SUMMARY_PATH, index=False)
forecast_source_summary.to_csv(FORECAST_SOURCE_SUMMARY_PATH, index=False)
missingness.to_csv(PANEL_MISSINGNESS_PATH, index=False)
missingness_by_group.to_csv(MISSINGNESS_BY_GROUP_PATH, index=False)
target_summary.to_csv(PANEL_TARGET_SUMMARY_PATH, index=False)
column_summary.to_csv(PANEL_COLUMN_SUMMARY_PATH, index=False)
leakage_check_df.to_csv(PANEL_LEAKAGE_CHECKS_PATH, index=False)
leakage_column_check.to_csv(LEAKAGE_COLUMN_CHECK_PATH, index=False)
for path in [
    ML_PANEL_PATH,
    FEATURE_REGISTRY_PATH,
    FEATURE_REGISTRY_AUDIT_PATH,
    INPUT_SCHEMA_AUDIT_PATH,
    SHAPE_SUMMARY_PATH,
    HORIZON_SUMMARY_PATH,
    MISSINGNESS_BY_GROUP_PATH,
    LEAKAGE_COLUMN_CHECK_PATH,
    FORECAST_SOURCE_SUMMARY_PATH,
]:
    print(f"Saved: {project_relative(path)}")

## Output summary and downstream use

Notebook 07 should consume `data/processed/ml_forecast_panel_full.parquet` and `outputs/ml_panel/ml_panel_feature_registry.csv`. These outputs define the modelling panel and registry-driven feature sets for the benchmark notebooks.


In [ ]:
print("Notebook 06 expected outputs:")
print(f"- {project_relative(ML_PANEL_PATH)}")
print(f"- {project_relative(FEATURE_REGISTRY_PATH)}")
print(f"- {project_relative(INPUT_SCHEMA_AUDIT_PATH)}")
print(f"- {project_relative(HORIZON_SUMMARY_PATH)}")
print(f"- {project_relative(FORECAST_SOURCE_SUMMARY_PATH)}")
print(f"- {project_relative(MISSINGNESS_BY_GROUP_PATH)}")
print(f"- {project_relative(LEAKAGE_COLUMN_CHECK_PATH)}")
print("Next notebook: notebooks/07_ml_benchmark_models.ipynb")